In [1]:
import pandas as pd
from scipy.stats import kendalltau
import rbo  # You must install this locally

# Load your CSV files
anova_chi2_df = pd.read_csv("p_values_for_base.csv")
value_dt = pd.read_csv("Booster_dt_shap_output.csv")
value_svc = pd.read_csv("Booster_svc_shap_output.csv")

anova_chi2_features = anova_chi2_df["Feature"].tolist()
value_features_dt = value_dt["Feature"].tolist()
value_features_svc = value_svc["Feature"].tolist()

# Prepare ranking dictionaries
anova_chi2_ranks = {f: i for i, f in enumerate(anova_chi2_features)}
value_ranks_dt = {f: i for i, f in enumerate(value_features_dt)}
value_ranks_svc = {f: i for i, f in enumerate(value_features_svc)}

# Common features for Kendall Tau between p and shap dt
common_features = list(set(anova_chi2_features) & set(value_features_dt))
ranks_1_a = [anova_chi2_ranks[f] for f in common_features]
ranks_2 = [value_ranks_dt[f] for f in common_features]

# Common features for Kendall Tau between p and shap svm rbf
common_features = list(set(anova_chi2_features) & set(value_features_svc))
ranks_1_b = [anova_chi2_ranks[f] for f in common_features]
ranks_3 = [value_ranks_svc[f] for f in common_features]

# Compute Kendall's Tau
kendall_tau_score, _ = kendalltau(ranks_1_a, ranks_2) # kendall's tau score for p and dt
kendall_tau_score_2, _ = kendalltau(ranks_1_b, ranks_3) # kendall's tau score for p and svm rbf

# Compute RBO
rbo_score_1 = rbo.RankingSimilarity(anova_chi2_features, value_features_dt).rbo(p=0.9)
rbo_score_2 = rbo.RankingSimilarity(anova_chi2_features, value_features_svc).rbo(p=0.9)

# Compute Recall@K
def recall_at_k(list1, list2, k):
    return len(set(list1[:k]).intersection(set(list2[:k]))) / k

recall_10 = recall_at_k(anova_chi2_features, value_features_dt, 10)
recall_20 = recall_at_k(anova_chi2_features, value_features_dt, 20)
recall_30 = recall_at_k(anova_chi2_features, value_features_dt, 30)

recall_10_2 = recall_at_k(anova_chi2_features, value_features_svc, 10)
recall_20_2 = recall_at_k(anova_chi2_features, value_features_svc, 20)
recall_30_2 = recall_at_k(anova_chi2_features, value_features_svc, 30)

# Display results

print("Scores between anova-chi (p-values) and decision tree shap values:")
print(f"Kendall's Tau: {kendall_tau_score:.3f}")
print(f"RBO (p=0.9): {rbo_score_1:.3f}")
print(f"Recall@10: {recall_10:.2f}")
print(f"Recall@20: {recall_20:.2f}")
print(f"Recall@30: {recall_30:.2f}")

print("Scores between anova-chi (p-values) and svm (rbf) shap values:")
print(f"Kendall's Tau: {kendall_tau_score_2:.3f}")
print(f"RBO (p=0.9): {rbo_score_2:.3f}")
print(f"Recall@10: {recall_10_2:.2f}")
print(f"Recall@20: {recall_20_2:.2f}")
print(f"Recall@30: {recall_30_2:.2f}")

Scores between anova-chi (p-values) and decision tree shap values:
Kendall's Tau: -0.159
RBO (p=0.9): 0.035
Recall@10: 0.00
Recall@20: 0.10
Recall@30: 0.10
Scores between anova-chi (p-values) and svm (rbf) shap values:
Kendall's Tau: -0.366
RBO (p=0.9): 0.017
Recall@10: 0.00
Recall@20: 0.05
Recall@30: 0.10
